In [549]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
from optbinning import ContinuousOptimalBinning

from sklearn.metrics import (
    mean_squared_error
)

In [550]:
df = pd.read_csv('src/data/training/processed/final_training_data.csv')

In [551]:
target_cols = df.filter(regex='^(dn__|ul__|mv__|madrid__|ep__|cb__|ob__|ab__)').columns

missing_or_zero = df[target_cols].isna() | (df[target_cols] == 0)

missing_fraction = missing_or_zero.mean(axis=1)

df = df[missing_fraction <= 0.5].copy()

In [552]:
df = df.drop(columns=['user_id','reference_date'])  # Drop non-feature columns

In [553]:
reg_feats = ['t__ccf', 'ob__open_limit_ref', 'ul__account_age_months']
clf_feats = ['ob__avg_util_ref', 'ob__open_limit_ref', 'ob__util_slope_3m', 'ob__max_util_0_1m', 'ab__days_down_6m', 'ab__max_one_day_drop_6m', 'ab__std_abs_change_6m', 'ab__mean_abs_change_6m', 'ab__std_util_0_1m', 'ab__near_zero_days_6m']

In [554]:
df = df[reg_feats]

In [555]:

# Columns to filter (all except 't__ccf')
cols_to_filter = df.columns.difference(['t__ccf'])

# Compute 0.9 quantile for these columns
quantiles = df[cols_to_filter].quantile(0.99)

# Filter rows where all selected columns <= 0.9 quantile
df_filtered = df[(df[cols_to_filter] <= quantiles).all(axis=1)]

# Keep the filtered columns + target
df_filtered = df_filtered[cols_to_filter.tolist() + ['t__ccf']]

print(f"Original shape: {df.shape}")
print(f"Filtered shape: {df_filtered.shape}")

Original shape: (12139, 3)
Filtered shape: (12023, 3)


In [556]:
df_filtered

,ob__open_limit_ref,ul__account_age_months,t__ccf
1,0.07324,25.0,1.11242
2,0.52000,8.0,1.00623
4,0.00190,54.0,1.03333
5,0.20496,23.0,0.00000
7,1.00000,9.0,0.00000
...,...,...,...
15610,0.96349,39.0,0.96832
15611,0.11277,15.0,0.00000
15613,0.77934,50.0,0.30497
15614,0.01024,53.0,1.02627


In [557]:
y = df_filtered[['t__ccf']].clip(lower=0, upper=1)
y = y.squeeze()          # converts DataFrame (n,1) -> Series (n,)
y = y.values.ravel()
X = df_filtered.drop(columns=['t__ccf'])

In [558]:
df_filtered

,ob__open_limit_ref,ul__account_age_months,t__ccf
1,0.07324,25.0,1.11242
2,0.52000,8.0,1.00623
4,0.00190,54.0,1.03333
5,0.20496,23.0,0.00000
7,1.00000,9.0,0.00000
...,...,...,...
15610,0.96349,39.0,0.96832
15611,0.11277,15.0,0.00000
15613,0.77934,50.0,0.30497
15614,0.01024,53.0,1.02627


In [559]:
def supervised_bin_and_dummy(X, y, max_bins=5):
    # force y to be 1D
    y = pd.Series(y).squeeze().values.ravel()

    binned_arrays = []
    binners = {}

    for col in X.columns:
        optb = ContinuousOptimalBinning(name=col, dtype="numerical", max_n_bins=max_bins)
        optb.fit(X[col], y)

        # Transform to bin index
        binned = optb.transform(X[col], metric="indices").astype(int).ravel()
        binned_arrays.append(binned)
        binners[col] = optb

    X_binned = pd.DataFrame(np.column_stack(binned_arrays), columns=X.columns)

    X_dummies = pd.get_dummies(X_binned, columns=X.columns, prefix=X.columns)

    return X_dummies, binners

In [560]:
X_dummies, binners = supervised_bin_and_dummy(X, y, max_bins=3)

In [561]:
X_dummies = X_dummies.astype(int)

In [562]:
X_train, X_test, y_train, y_test = train_test_split(X_dummies, y, test_size=0.2, random_state=42)

In [563]:
binners

{'ob__open_limit_ref': ContinuousOptimalBinning(max_n_bins=3, name='ob__open_limit_ref'),
 'ul__account_age_months': ContinuousOptimalBinning(max_n_bins=3, name='ul__account_age_months')}

In [564]:
X_train

,ob__open_limit_ref_0,ob__open_limit_ref_1,ob__open_limit_ref_2,ul__account_age_months_0,ul__account_age_months_1,ul__account_age_months_2
2140,1,0,0,1,0,0
5727,0,0,1,1,0,0
5922,1,0,0,1,0,0
909,0,0,1,1,0,0
8968,1,0,0,1,0,0
...,...,...,...,...,...,...
11964,1,0,0,1,0,0
5191,1,0,0,1,0,0
5390,1,0,0,1,0,0
860,0,0,1,1,0,0


In [565]:
#X_train_sm = sm.add_constant(X_train)
model = sm.OLS(y_train, X_train).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.005
Model:                            OLS   Adj. R-squared:                  0.005
Method:                 Least Squares   F-statistic:                     10.36
Date:                Wed, 10 Dec 2025   Prob (F-statistic):           6.35e-10
Time:                        14:55:10   Log-Likelihood:                -6185.3
No. Observations:                9618   AIC:                         1.238e+04
Df Residuals:                    9612   BIC:                         1.243e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
ob__open_limit_ref_0    

In [391]:
X_train

,ob__avg_util_0_1m_0,ob__avg_util_0_1m_1,ob__avg_util_0_1m_2,ob__avg_util_0_1m_3
909,1,0,0,0
7471,1,0,0,0
496,0,0,1,0
9168,1,0,0,0
6842,1,0,0,0
...,...,...,...,...
11964,1,0,0,0
5191,1,0,0,0
5390,0,0,1,0
860,1,0,0,0
